# 01 — Eksplorasi Dataset Resep

## Tujuan
Notebook ini digunakan untuk memahami struktur dan kualitas dataset resep yang akan digunakan pada proyek DapurFinder.

## Tahapan
1. Memuat dataset resep.
2. Memeriksa ukuran dan nama kolom.
3. Melihat beberapa contoh data.
4. Memeriksa data kosong dan duplikasi.
5. Memeriksa jumlah resep per kategori.
6. Menyusun kesimpulan awal untuk tahap preprocessing.

## Dataset
Dataset berisi informasi judul resep, bahan, langkah memasak, jumlah suka, URL, dan kategori masakan.

In [1]:
from pathlib import Path
import pandas as pd

# Menentukan lokasi dataset berdasarkan folder proyek
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_PATH = PROJECT_DIR / "dataset" / "recipes.csv"

# Memastikan file tersedia sebelum dibaca
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan di: {DATASET_PATH}\n"
        "Pastikan recipes.csv berada di folder dataset."
    )

df = pd.read_csv(DATASET_PATH)

print("Dataset berhasil dimuat.")
print(f"Lokasi file : {DATASET_PATH}")
print(f"Jumlah baris: {df.shape[0]:,}")
print(f"Jumlah kolom: {df.shape[1]}")

Dataset berhasil dimuat.
Lokasi file : D:\STKI-Recipe-Finder\dataset\recipes.csv
Jumlah baris: 15,641
Jumlah kolom: 6


In [2]:
# Memeriksa nama kolom dan tipe data
print("Nama kolom:")
print(df.columns.tolist())

print("\nTipe data:")
display(df.dtypes.to_frame(name="Tipe Data"))

print("\nLima contoh resep:")
display(df.head())

Nama kolom:
['Title', 'Ingredients', 'Steps', 'Loves', 'URL', 'Category']

Tipe data:


,Tipe Data
Title,object
Ingredients,object
Steps,object
Loves,int64
URL,object
Category,object



Lima contoh resep:


,Title,Ingredients,Steps,Loves,URL,Category
0,Ayam Woku Manado,1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk ...,Cuci bersih ayam dan tiriskan. Lalu peras jeru...,1,/id/resep/4473027-ayam-woku-manado,ayam
1,Ayam goreng tulang lunak,1 kg ayam (dipotong sesuai selera jangan kecil...,"Haluskan bumbu2nya (BaPut, ketumbar, kemiri, k...",1,/id/resep/4471956-ayam-goreng-tulang-lunak,ayam
2,Ayam cabai kawin,1/4 kg ayam--3 buah cabai hijau besar--7 buah ...,Panaskan minyak di dalam wajan. Setelah minyak...,2,/id/resep/4473057-ayam-cabai-kawin,ayam
3,Ayam Geprek,250 gr daging ayam (saya pakai fillet)--Secuku...,Goreng ayam seperti ayam krispi--Ulek semua ba...,10,/id/resep/4473023-ayam-geprek,ayam
4,Minyak Ayam,400 gr kulit ayam & lemaknya--8 siung bawang p...,Cuci bersih kulit ayam. Sisihkan--Ambil 50 ml ...,4,/id/resep/4427438-minyak-ayam,ayam


In [3]:
# Menghitung data kosong
ringkasan_kosong = pd.DataFrame({
    "Jumlah Kosong": df.isna().sum(),
    "Persentase Kosong (%)": (
        df.isna().mean() * 100
    ).round(2)
})

display(ringkasan_kosong)

,Jumlah Kosong,Persentase Kosong (%)
Title,0,0.00
Ingredients,48,0.31
Steps,46,0.29
Loves,0,0.00
URL,0,0.00
Category,0,0.00


In [4]:
# Memeriksa jumlah baris duplikat dan jumlah resep per kategori
print("Jumlah baris duplikat:", df.duplicated().sum())

print("\nJumlah resep per kategori:")
display(
    df["Category"]
    .value_counts(dropna=False)
    .rename_axis("Kategori")
    .reset_index(name="Jumlah Resep")
)

Jumlah baris duplikat: 0

Jumlah resep per kategori:


,Kategori,Jumlah Resep
0,udang,1994
1,tempe,1986
2,tahu,1985
3,telur,1974
4,sapi,1958
5,ikan,1932
6,ayam,1916
7,kambing,1896


## Kesimpulan Eksplorasi

Dataset DapurFinder terdiri dari 15.641 resep dan enam kolom, yaitu Title, Ingredients, Steps, Loves, URL, dan Category. Data mencakup delapan kategori masakan.

Pemeriksaan awal menemukan 48 nilai kosong pada kolom Ingredients dan 46 nilai kosong pada kolom Steps. Tidak ditemukan baris yang sepenuhnya duplikat.

Kolom Title dan Ingredients akan menjadi informasi utama dalam proses pencarian resep. Data kosong akan ditangani pada tahap preprocessing agar tidak mengganggu proses pencarian. Kolom Category dapat digunakan untuk penyaringan kategori, sedangkan URL dapat digunakan untuk merujuk ke sumber resep.

In [5]:
# Memeriksa teks kosong atau hanya berisi spasi
kolom_teks = ["Title", "Ingredients", "Steps", "URL", "Category"]

for kolom in kolom_teks:
    teks = df[kolom].astype("string")
    jumlah_kosong = (
        teks.isna() | teks.str.strip().eq("")
    ).sum()

    print(f"{kolom}: {jumlah_kosong} nilai kosong atau hanya spasi")

# Memeriksa judul resep yang berulang
jumlah_judul_berulang = df["Title"].duplicated().sum()

print("\nJumlah judul yang berulang setelah kemunculan pertama:",
      jumlah_judul_berulang)

# Melihat contoh judul yang berulang, jika ada
judul_berulang = (
    df.loc[df["Title"].duplicated(keep=False), "Title"]
    .value_counts()
    .head(10)
)

display(judul_berulang)

Title: 0 nilai kosong atau hanya spasi
Ingredients: 48 nilai kosong atau hanya spasi
Steps: 46 nilai kosong atau hanya spasi
URL: 0 nilai kosong atau hanya spasi
Category: 0 nilai kosong atau hanya spasi

Jumlah judul yang berulang setelah kemunculan pertama: 3170


Title
Tongseng Kambing    55
[Notitle]           46
Beef Teriyaki       37
Tongseng kambing    31
Tempe mendoan       30
Sapi Lada Hitam     28
Sate Kambing        25
Tempe Mendoan       25
Perkedel tahu       24
Telur Balado        23
Name: count, dtype: int64

In [6]:
# 1. Memeriksa judul yang tidak valid
judul_tidak_valid = df[
    df["Title"].str.strip().str.casefold().isin(
        ["[notitle]", "notitle", ""]
    )
]

print("Jumlah resep dengan judul [Notitle]:", len(judul_tidak_valid))
display(judul_tidak_valid[["Title", "Ingredients", "Steps", "Category"]].head(5))


# 2. Menghitung judul berulang setelah normalisasi huruf
judul_normal = df["Title"].str.strip().str.casefold()

print(
    "\nJumlah baris dengan judul yang berulang "
    "(mengabaikan kapitalisasi):",
    judul_normal.duplicated().sum()
)


# 3. Memeriksa resep dengan judul, bahan, dan langkah yang sama
kolom_resep = ["Title", "Ingredients", "Steps"]

resep_normal = df[kolom_resep].copy()

for kolom in kolom_resep:
    resep_normal[kolom] = (
        resep_normal[kolom]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
    )

duplikat_resep = resep_normal.duplicated(keep=False)

print(
    "\nJumlah baris yang memiliki resep identik "
    "(judul, bahan, dan langkah):",
    int(duplikat_resep.sum())
)

display(df.loc[duplikat_resep, kolom_resep + ["Category"]].head(10))

Jumlah resep dengan judul [Notitle]: 46


,Title,Ingredients,Steps,Category
255,[Notitle],NaN,NaN,ayam
258,[Notitle],NaN,NaN,ayam
298,[Notitle],NaN,NaN,ayam
302,[Notitle],NaN,NaN,ayam
464,[Notitle],NaN,NaN,ayam



Jumlah baris dengan judul yang berulang (mengabaikan kapitalisasi): 3915

Jumlah baris yang memiliki resep identik (judul, bahan, dan langkah): 1325


,Title,Ingredients,Steps,Category
79,"Semur Ati Ayam, Telur Puyuh, Tahu","5 buah ati ayam, cuci bersih--6 Butir telur pu...","Panaskan minyak goreng. Tumis bumbu halus,sala...",ayam
87,Siomay Ayam Udang,"150 gr ayam, blender--150 gr udang cincang kas...",Campur semua bahan aduk rata.--Oles minyak kul...,ayam
101,Ayam Dan Tempe Goreng Serundeng,1 dada ayam--1/2 kelapa parut--2 tempe (potong...,"Setelah ayam dicuci bersih, didihkan air. Masu...",ayam
143,Tahu ayam udang,3 bh Tahu cina putih--100 gr udang dikupas--20...,Potong tahu--Siapkan udang dan ayam--Haluskan ...,ayam
156,"Semur Ayam, Tahu, Kentang",1/4 kg Sayap ayam--2 buah kentang sedang--1 ko...,Haluskan bawang merah dan bawang putih. Tumis ...,ayam
157,Tongseng ayam tahu,500 gr ayam dipotong kecil2--2 bh tahu dipoton...,Haluskan bumbu. Kemudian tumis bumbu sampai wa...,ayam
180,Siomay Ayam Udang,25 lembar kulit pangsit--250 gr ayam giling (s...,"Campur daging ayam, udang, labu siam, wortel, ...",ayam
226,Rolade Ayam Tahu,1/2 kg daging ayam fillet--2 buah tahu putih u...,"Rajang kasar bawang putih, bawang merah dan ba...",ayam
255,[Notitle],NaN,NaN,ayam
258,[Notitle],NaN,NaN,ayam


In [7]:
# Salinan untuk simulasi pembersihan, tanpa mengubah df asli
df_simulasi = df.copy()

# 1. Mengeluarkan resep tanpa judul, bahan, atau langkah yang valid
kolom_wajib = ["Title", "Ingredients", "Steps"]

for kolom in kolom_wajib:
    df_simulasi[kolom] = df_simulasi[kolom].replace(
        r"^\s*$", pd.NA, regex=True
    )

jumlah_awal = len(df_simulasi)

df_simulasi = df_simulasi.dropna(subset=kolom_wajib).copy()

print("Jumlah data awal:", jumlah_awal)
print("Resep tanpa judul/bahan/langkah yang dikeluarkan:",
      jumlah_awal - len(df_simulasi))
print("Jumlah setelah memeriksa informasi wajib:", len(df_simulasi))


# 2. Membuat kunci pembanding yang dinormalisasi
kolom_resep = ["Title", "Ingredients", "Steps"]

kunci_resep = df_simulasi[kolom_resep].copy()

for kolom in kolom_resep:
    kunci_resep[kolom] = (
        kunci_resep[kolom]
        .astype(str)
        .str.strip()
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
    )

# 3. Menghitung dampak jika resep identik disisakan satu
duplikat = kunci_resep.duplicated(keep="first")

print("\nJumlah resep identik yang dapat dikurangi:", int(duplikat.sum()))
print("Jumlah resep unik setelah simulasi:", len(df_simulasi) - int(duplikat.sum()))

Jumlah data awal: 15641
Resep tanpa judul/bahan/langkah yang dikeluarkan: 48
Jumlah setelah memeriksa informasi wajib: 15593

Jumlah resep identik yang dapat dikurangi: 650
Jumlah resep unik setelah simulasi: 14943


## Kesimpulan dan Rencana Preprocessing

Dataset awal berisi 15.641 resep dengan enam kolom. Pemeriksaan menemukan 48 baris dengan informasi wajib yang tidak lengkap. Setelah baris tersebut dikeluarkan dalam simulasi, tersisa 15.593 resep.

Pemeriksaan duplikasi menggunakan kombinasi judul, bahan, dan langkah memasak yang telah dinormalisasi menunjukkan bahwa 650 baris merupakan duplikat tambahan setelah kemunculan pertama. Dengan menyisakan satu salinan untuk setiap kombinasi resep identik, jumlah data dalam simulasi menjadi 14.943 resep.

Pada tahap preprocessing berikutnya, data akan dibersihkan dan dinormalisasi untuk mendukung pencarian resep. Dataset asli tetap dipertahankan sebagai sumber data. Hasil preprocessing akan disimpan sebagai data terpisah agar prosesnya dapat diperiksa dan diulang.